# Employee Salary Management System:


### Step 1:

In [12]:
import json
from pathlib import Path

DATA_FILE = Path("employees.json")
print("Data file location:", DATA_FILE.resolve())

Data file location: E:\Saylani AI and data science course\Batch 16 - 2026\classes\Master classes\Employee Salary management system (OOP)\employees.json


### Step 2:

The class stores each employee's information and contains methods to calculate salary.
- __init__() is the constructor.
- self refers to the current object.
- calculate_gross_salary() adds basic salary and allowances.
- calculate_net_salary() subtracts deductions.
- validate() checks the employee data.
- to_dict() converts the object to a dictionary for JSON storage.
- from_dict() reconstructs an object from a saved dictionary.

In [13]:
class Employee():
    # constructor
    def __init__(self, employee_id, name, designation, basic_salary, allowances, deductions):
        # instance attributes
        self.employee_id = employee_id
        self.name = name
        self.designation = designation
        self.basic_salary = basic_salary
        self.allowances = allowances
        self.deductions = deductions

    # instance methods
    def calculate_gross_salary(self):
        return self.basic_salary + self.allowances

    def calculate_net_salary(self):
        return self.calculate_gross_salary() - self.deductions

    def validate(self):
        if not self.employee_id:
            raise ValueError("Employee ID cannot be empty.")

        if not self.name:
            raise ValueError("Employee name cannot be empty.")

        if not self.designation:
            raise ValueError("Designation cannot be empty.")

        amounts = (
            self.basic_salary,
            self.allowances,
            self.deductions
        )

        if any(amount < 0 for amount in amounts):
            raise ValueError("Amounts cannot be negative.")

        if self.deductions > self.calculate_gross_salary():
            raise ValueError("Deductions cannot exceed gross salary.")

        return True

    def to_dict(self):
        return {
            "employee_id": self.employee_id,
            "name": self.name,
            "designation": self.designation,
            "basic_salary": self.basic_salary,
            "allowances": self.allowances,
            "deductions": self.deductions
        }

    @classmethod
    def from_dict(cls, data):
        employee = cls(
            data["employee_id"],
            data["name"],
            data["designation"],
            data["basic_salary"],
            data["allowances"],
            data["deductions"]
        )

        return employee

    def display_salary_slip(self):
        print("\n" + "=" * 40)
        print("             SALARY SLIP")
        print("=" * 40)
        print(f"Employee ID  : {self.employee_id}")
        print(f"Name         : {self.name}")
        print(f"Designation  : {self.designation}")
        print("-" * 40)
        print(f"Basic Salary : Rs. {self.basic_salary:,.2f}")
        print(f"Allowances   : Rs. {self.allowances:,.2f}")
        print(f"Gross Salary : Rs. {self.calculate_gross_salary():,.2f}")
        print(f"Deductions   : Rs. {self.deductions:,.2f}")
        print("-" * 40)
        print(f"NET SALARY   : Rs. {self.calculate_net_salary():,.2f}")
        print("=" * 40)

### Step 3:

An object is an instance of a class. Run this cell to create an example employee and calculate the salary.

In [14]:
emp1 = Employee("EMP001", "Ali Ahmed", "Software Developer", 50000, 10000, 5000)

emp1.validate()

print("Employee:", emp1.name)
print("Gross salary: Rs.", emp1.calculate_gross_salary())
print("Net salary: Rs.", emp1.calculate_net_salary())

emp1.display_salary_slip()

Employee: Ali Ahmed
Gross salary: Rs. 60000
Net salary: Rs. 55000

             SALARY SLIP
Employee ID  : EMP001
Name         : Ali Ahmed
Designation  : Software Developer
----------------------------------------
Basic Salary : Rs. 50,000.00
Allowances   : Rs. 10,000.00
Gross Salary : Rs. 60,000.00
Deductions   : Rs. 5,000.00
----------------------------------------
NET SALARY   : Rs. 55,000.00


### Step 4: Build Manager class

This class manages the employee objects and handles saving/loading JSON data. It also provides methods to add, search, list, and display salary slips.

In [27]:
class SalaryManagementSystem:
    def __init__(self, data_file=DATA_FILE):
        self.data_file = Path(data_file)       # data file path
        self.employees = []                    # employees records list
        self.load_data()                       # load employees data

    def load_data(self):
        if not self.data_file.exists():
            self.employees = []
            return

        try:
            with self.data_file.open("r", encoding="utf-8") as file:
                records = json.load(file)

            if not isinstance(records, list):
                raise ValueError("Employee data must be a JSON list.")

            self.employees = [Employee.from_dict(record) for record in records]

        except (OSError, json.JSONDecodeError, ValueError, TypeError, KeyError) as error:
            raise RuntimeError(f"Could not load employee data: {error}") from error

    def save_data(self):
        records = [employee.to_dict() for employee in self.employees]
        with self.data_file.open("w", encoding="utf-8") as file:
            json.dump(records, file, indent=4, ensure_ascii=False)

    def add_employee(self):
        # get employee details
        employee_id = input("Enter Employee ID: ").casefold()
        name = input("Enter Name: ")
        designation = input("Enter Designation: ")
        basic_salary = float(input("Enter Basic Salary: "))
        allowances = float(input("Enter Allowance: "))
        deductions = float(input("Enter Deductions: "))

        # create Employee Object
        employee = Employee(employee_id, name, designation, basic_salary, allowances, deductions)

        # Validate employee Data
        employee.validate()

        # Check if employee Exists
        if self.search_employee(employee.employee_id):
            raise ValueError("This employee ID already exists.")

        self.employees.append(employee)
        try:
            self.save_data()
        except OSError:
            self.employees.pop()
            raise

        return employee

    def view_all_employees(self):
        if not self.employees:
            print("No employee records found.")
            return

        print("\n--- Employee Records ---")
        for employee in self.employees:
            print(
                f"{employee.employee_id} | {employee.name} | "
                f"{employee.designation} | "
                f"Net Salary: Rs. {employee.calculate_net_salary():,.2f}"
            )

    def search_employee(self, employee_id):
        for employee in self.employees:
            if employee.employee_id.casefold() == employee_id.strip().casefold():
                return employee
        return None
        
    def generate_salary_slip(self, employee_id):
        employee = self.search_employee(employee_id)
        if employee is None:
            print("Employee not found.")
            return
        employee.display_salary_slip()

    def update_employee(self, employee_id):
        employee = self.search_employee(employee_id)
        if employee is None:
            print("Employee does not exist")
            return

        new_designation = input("Enter new Designation: ")
        new_basic_salary = float(input("Enter new Basic Salary: "))
        new_allowances = float(input("Enter new allowances: "))
        new_deductions = float(input("Enter new deductions: "))

        # Create a temporary object to validate the new data
        updated_employee = Employee(
            employee.employee_id,
            employee.name,
            new_designation,
            new_basic_salary,
            new_allowances,
            new_deductions
        )

        updated_employee.validate()

        # Store the old values in case saving fails
        old_values = (
            employee.designation,
            employee.basic_salary,
            employee.allowances,
            employee.deductions
        )

        # Update employee information
        employee.designation = new_designation
        employee.basic_salary = new_basic_salary
        employee.allowances = new_allowances
        employee.deductions = new_deductions

        try:
            self.save_data()
        except OSError as error:
            # Restore the original values if saving fails
            (
                employee.designation,
                employee.basic_salary,
                employee.allowances,
                employee.deductions
            ) = old_values

            print(f"Error saving employee data: {error}")
            return

        print("Employee updated successfully!")
        print(f"Updated Designation: {employee.designation}")
        print(f"Updated Gross Salary: Rs. {employee.calculate_gross_salary():,.2f}")
        print(f"Updated Net Salary: Rs. {employee.calculate_net_salary():,.2f}")


In [28]:
ESMS = SalaryManagementSystem()

In [30]:
while True:
    print("-" * 45)
    print("WELCOME TO EMPLOYEE SALARY MANAGEMENT SYSTEM")
    print("=" * 45)
    print("1. Enter {1} to ADD Employee: ")
    print("2. Enter {2} to VIEW All Employees: ")
    print("3. Enter {3} to SEARCH Employee: ")
    print("4. Enter {4} to Generate Salary Slip: ")
    print("5. Enter {5} to Update Employee: ")
    print("6. Enter {6} to Exit: ")

    print("-" * 45)
    choice = input("Enter your choice: ")

    if choice == "1":
        ESMS.add_employee()
    elif choice == "2":
        ESMS.view_all_employees()
    elif choice == "3":
        employee_id = input("Enter ID to Search employee: ")
        employee = ESMS.search_employee(employee_id)

        if employee:
            print(
                f"{employee.employee_id} | {employee.name} | "
                f"{employee.designation} | "
                f"Net Salary: Rs. {employee.calculate_net_salary():,.2f}"
            )
        else:
            print("Employee Does not exist")
    elif choice == "4":
        employee_id = input("Enter ID to get Salary Slip: ")
        ESMS.generate_salary_slip(employee_id)
    elif choice == "5":
        employee_id = input("Enter ID to get Salary Slip: ")
        ESMS.update_employee(employee_id)
    elif choice == "6":
        break
    else:
        print("Invalid Choice")

---------------------------------------------
WELCOME TO EMPLOYEE SALARY MANAGEMENT SYSTEM
1. Enter {1} to ADD Employee: 
2. Enter {2} to VIEW All Employees: 
3. Enter {3} to SEARCH Employee: 
4. Enter {4} to Generate Salary Slip: 
5. Enter {5} to Update Employee: 
6. Enter {6} to Exit: 
---------------------------------------------


Enter your choice:  3
Enter ID to Search employee:  e1-009


e1-009 | Ahsan | Lecturer | Net Salary: Rs. 105,000.00
---------------------------------------------
WELCOME TO EMPLOYEE SALARY MANAGEMENT SYSTEM
1. Enter {1} to ADD Employee: 
2. Enter {2} to VIEW All Employees: 
3. Enter {3} to SEARCH Employee: 
4. Enter {4} to Generate Salary Slip: 
5. Enter {5} to Update Employee: 
6. Enter {6} to Exit: 
---------------------------------------------


Enter your choice:  2



--- Employee Records ---
e1-009 | Ahsan | Lecturer | Net Salary: Rs. 105,000.00
e1-088 | Saad | instructor | Net Salary: Rs. 66,000.00
---------------------------------------------
WELCOME TO EMPLOYEE SALARY MANAGEMENT SYSTEM
1. Enter {1} to ADD Employee: 
2. Enter {2} to VIEW All Employees: 
3. Enter {3} to SEARCH Employee: 
4. Enter {4} to Generate Salary Slip: 
5. Enter {5} to Update Employee: 
6. Enter {6} to Exit: 
---------------------------------------------


Enter your choice:  5
Enter ID to get Salary Slip:  e1-009
Enter new Designation:  Senior Lecturer
Enter new Basic Salary:  100000
Enter new allowances:  50000
Enter new deductions:  20000


Employee updated successfully!
Updated Designation: Senior Lecturer
Updated Gross Salary: Rs. 150,000.00
Updated Net Salary: Rs. 130,000.00
---------------------------------------------
WELCOME TO EMPLOYEE SALARY MANAGEMENT SYSTEM
1. Enter {1} to ADD Employee: 
2. Enter {2} to VIEW All Employees: 
3. Enter {3} to SEARCH Employee: 
4. Enter {4} to Generate Salary Slip: 
5. Enter {5} to Update Employee: 
6. Enter {6} to Exit: 
---------------------------------------------


Enter your choice:  6
